In [44]:
##Step 0: Install theses necesary packages
##Install google-api-python-client google-auth google-auth-oauthlib pandas beautifulsoup4 gspread
##install -q pillow

##Step 1: Import packages after installing
#Import for step 2
import os
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import gspread
#Import for step 3
import re, unicodedata
import pandas as pd
#Import for step 5
import re
from urllib.parse import urlparse, parse_qs
#Import for step 6
import re
from googleapiclient.http import MediaInMemoryUpload
#Import for step 7
import os
from bs4 import BeautifulSoup
#Import for step 8
import base64
import io
import requests
#Import for step 9
import unicodedata
#Import for step 10
from urllib.parse import urlparse
#Import for step 11
from PIL import Image, UnidentifiedImageError
#Import for step 14
from bs4 import BeautifulSoup
import re
#Import for step 17
from bs4 import BeautifulSoup, NavigableString
import html
#Import for step 18
from bs4 import BeautifulSoup, NavigableString
from urllib.parse import urlparse, parse_qs, unquote
import re
#Import dotenv
import dotenv
dotenv.load_dotenv()

True

In [45]:
##Step 2: Build the drive client

#Scopes: read-only for both Drive & Sheets
SCOPES = [
    "https://www.googleapis.com/auth/drive.file",
    "https://www.googleapis.com/auth/drive.readonly",
    "https://www.googleapis.com/auth/spreadsheets.readonly",
]
CLIENT_SECRETS = "client_secret.json"   # <-- must exist locally (Desktop OAuth)
TOKEN_PATH = "token.json"               # will be created on first run

#Import get credentials function
from functions import get_creds
creds=get_creds()

# Build API clients
drive = build("drive", "v3", credentials=creds)
gc = gspread.authorize(creds)

print("Drive & Sheets clients ready")

Drive & Sheets clients ready


In [46]:
## Step 3: Read the sheet and return docs_url

## Define the sheets for getting rows
SHEET_URL = "https://docs.google.com/spreadsheets/d/1sMgzVgxNzT5RcYHlrDlQkNKYkfwTWxiys1vYhNWF38Q/edit?gid=1887291779#gid=1887291779"
TAB_NAME  = "DSBV | Rackcosmo" 

## Get dataframe of all rows in sheet
ws = gc.open_by_url(SHEET_URL).worksheet(TAB_NAME)
rows = ws.get_all_values()
df = pd.DataFrame(rows[1:], columns=rows[0]) if rows else pd.DataFrame()
df.head()

## Reload functions module for updates
import importlib, functions
importlib.reload(functions) 

## Get the slug format of main keyword later used for creating empty post
from functions import strip_diacritics  #normalize words into base characters
from functions import slugify           #return the slugified format of any words after normalizing
from functions import find_col          #find collumn with exact name

## Check if the dataframe is empty
if df.empty:
    raise RuntimeError("DataFrame `df` is empty. Run Step 3 first.")

# Find the main keyword column
mk_col = find_col(df.columns, ["main keyword", "Main Keyword", "main_keyword"])
if not mk_col:
    raise ValueError('Could not find a "main keyword" column in the sheet.')

# Add a slug column to dataframe from the "main keyword" column
df["main_keyword_slug"] = df[mk_col].map(slugify)

# Optional: build a separate list if you prefer not to rely on df later
main_keyword_slugs = [
    {
        "row_index": i + 2,  # head=1 => first data row is sheet row 2
        "main_keyword": str(val or "").strip(),
        "main_keyword_slug": slugify(val),
    }
    for i, val in enumerate(df[mk_col].tolist())
    if str(val or "").strip()
]

print(f"Created slugs for {len(main_keyword_slugs)} row(s).")
# Quick peek:
# df[["{mk_col}", "main_keyword_slug"]].head()
# main_keyword_slugs[:3]
df.head()

Created slugs for 34 row(s).


,STT,main keyword,Link docs Bài viết,Categories,Tags,Duyệt đăng,Trạng thái,Link nháp,main_keyword_slug
0,1,test Rackcosmo,https://docs.google.com/document/d/1tA_4WjRQ6m...,,,TRUE,Đã đăng tự động,https://rackcosmo.com/?p=1917,test-rackcosmo
1,2,kệ drive in,https://docs.google.com/document/d/1xP3fLHbpW5...,,,FALSE,,,ke-drive-in
2,3,kệ công nghiệp,https://docs.google.com/document/d/1I9cwdSrHb0...,,,FALSE,,,ke-cong-nghiep
3,4,kệ kho hàng,https://docs.google.com/document/d/1VrA5LaY7IE...,,,FALSE,,,ke-kho-hang
4,5,Kệ trung tải,https://docs.google.com/document/d/14lV24IGJAd...,,,FALSE,,,ke-trung-tai


In [47]:
## Step 4: Get docs_url_to_run

# Define necessary variables
URL_COLUMN_NAME = "Link docs Bài viết"       # your URL column
SLUG_COLUMN_NAME = "main_keyword_slug"      # newly added slug column

## Check if neccessary columns are available 
df.columns = [c.strip() for c in df.columns]  #Strip out spaces in columns
for col in ["Duyệt đăng", "Trạng thái", URL_COLUMN_NAME, SLUG_COLUMN_NAME]:
    if col not in df.columns:
        raise ValueError(f"Missing column: {col}. Found: {list(df.columns)}")

## Reload functions module for updates
import importlib, functions
importlib.reload(functions) 

# Import functions used for checking if an url is checked for running but has not been run.
from functions import is_checked
from functions import is_blank

## Get rows sastisfiying the wanted rules
mask = df["Duyệt đăng"].apply(is_checked) & df["Trạng thái"].apply(is_blank)        # Define filtering rule
filtered = df.loc[mask, [URL_COLUMN_NAME, SLUG_COLUMN_NAME]].copy()             # keep URL + slug columns
filtered[URL_COLUMN_NAME] = filtered[URL_COLUMN_NAME].astype(str).str.strip().replace({"": pd.NA})      # clean URLs and dedup by URL to keep alignment
clean = (
    filtered
    .dropna(subset=[URL_COLUMN_NAME])
    .drop_duplicates(subset=[URL_COLUMN_NAME], keep="first")
)

# return outputs
docs_url_to_run = clean[URL_COLUMN_NAME].tolist()
main_keyword_slug_to_run = clean[SLUG_COLUMN_NAME].astype(str).str.strip().tolist()
docs_to_run = [
    {"url": u, "slug": s}
    for u, s in zip(docs_url_to_run, main_keyword_slug_to_run)
]
print(f"docs_url_to_run → {len(docs_url_to_run)}")
print(f"main_keyword_slug_to_run → {len(main_keyword_slug_to_run)}")

# Quick check
print(docs_to_run)


docs_url_to_run → 1
main_keyword_slug_to_run → 1
[{'url': 'https://docs.google.com/document/d/19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E/edit?usp=sharing', 'slug': 'ke-trung-tai-4-tang'}]


In [48]:
## Step 5: Export docs to html
# Export Google Docs → HTML (in-memory, no saving)

#Check the drive client
try:
    drive  # use existing Drive client
except NameError:
    raise RuntimeError("Drive client `drive` not found. Run the auth cell first.")

## Reload functions module for updates
import importlib, functions
importlib.reload(functions) 

## Export html files from docs id
from functions import extract_file_id
from functions import export_doc_html_bytes

# Check if docs_url_to_run is available
try:
    docs_url_to_run  # list of URLs prepared earlier
except NameError:
    raise RuntimeError("`docs_url_to_run` is not defined. Build it from your sheet first.")

# Create list for storing html files
exported_html_docs = []  # list of dicts: {url, file_id, name, html} (html is a UTF-8 string)

# Downloading docs in html
for url in docs_url_to_run:
    try:
        fid = extract_file_id(url)
        html_bytes, meta = export_doc_html_bytes(fid)
        html_text = html_bytes.decode("utf-8", errors="ignore")   # <<< define html_text

        exported_html_docs.append({
            "url": url,
            "file_id": fid,
            "name": meta["name"],
            "html": html_text
        })

        print(f"Exported: {meta['name']}")
    except TypeError as e:
        print(f"Skipping (not a Google Doc): {url} | {e}")
    except Exception as e:
        print(f"Failed: {url} | {e}")

print(f"\nDone. Exported {len(exported_html_docs)} Google Doc(s). ")
print(exported_html_docs)

Exported: Bài viết | 36 - kệ trung tải 4 tầng

Done. Exported 1 Google Doc(s). 
[{'url': 'https://docs.google.com/document/d/19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E/edit?usp=sharing', 'file_id': '19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E', 'name': 'Bài viết | 36 - kệ trung tải 4 tầng', 'html': '<html><head><meta content="text/html; charset=UTF-8" http-equiv="content-type"><style type="text/css"> ul.lst-kix_sisju2akkx7v-7{list-style-type:none}.lst-kix_ksn62bjcvjdf-5 > li:before{content:"■  "}ul.lst-kix_sisju2akkx7v-8{list-style-type:none}ul.lst-kix_sisju2akkx7v-5{list-style-type:none}.lst-kix_ksn62bjcvjdf-4 > li:before{content:"○  "}.lst-kix_ksn62bjcvjdf-6 > li:before{content:"●  "}ul.lst-kix_sisju2akkx7v-6{list-style-type:none}ul.lst-kix_sisju2akkx7v-3{list-style-type:none}ul.lst-kix_sisju2akkx7v-4{list-style-type:none}ul.lst-kix_sisju2akkx7v-1{list-style-type:none}ul.lst-kix_sisju2akkx7v-2{list-style-type:none}ul.lst-kix_ipv0ujbgrru9-8{list-style-type:none}ul.lst-kix_ipv0ujbgr

In [49]:
## Step 6: Upload exported HTML to Google Drive

#Check if drive is available
try:
    drive  # Drive client from your auth step
    exported_html_docs  # list from Step 5
except NameError:
    raise RuntimeError("Missing `drive` or `exported_html_docs`. Run Step 5 first.")

# Load drive_folder_id
dotenv.load_dotenv()
DRIVE_FOLDER_ID

## Reload functions module for updates
import importlib, functions
importlib.reload(functions) 
from functions import safe_filename

# Create list for storing uploaded html files
uploaded_html_files = []  # will hold dicts: {file_id, name, webViewLink, source_url, source_file_id}

for item in exported_html_docs:
    html_text = item["html"]
    base = safe_filename(item["name"])
    filename = f"{base}.html"

    media = MediaInMemoryUpload(
        html_text.encode("utf-8"),
        mimetype="text/html",
        resumable=False,
    )

    metadata = {
        "name": filename,
        "parents": [DRIVE_FOLDER_ID],
        "mimeType": "text/html",
    }

    try:
        file = (
            drive.files()
            .create(body=metadata, media_body=media, fields="id,name,webViewLink")
            # If uploading to a Shared Drive, uncomment the next line:
            # .create(body=metadata, media_body=media, fields="id,name,webViewLink", supportsAllDrives=True)
            .execute()
        )

        uploaded_html_files.append({
            "file_id": file["id"],
            "name": file["name"],
            "webViewLink": file.get("webViewLink"),
            "source_url": item["url"],
            "source_file_id": item["file_id"],
        })

        print(f"Uploaded: {file['name']}  →  {file['webViewLink']}")
    except Exception as e:
        print(f"Failed to upload {filename}: {e}")

print(f"\nDone. Uploaded {len(uploaded_html_files)} HTML file(s) to Drive folder {DRIVE_FOLDER_ID}.")


Uploaded: Bài-viết-_-36-kệ-trung-tải-4-tầng.html  →  https://drive.google.com/file/d/1etG6MNJMosw63yf86pXY_lIEErq2OqAu/view?usp=drivesdk

Done. Uploaded 1 HTML file(s) to Drive folder 1NjOhY5tEOFUrz49YeQiA_fwGzKg_4CJX.


In [50]:
## Step 7: Process each local HTML file and extract <img> tags
from bs4 import BeautifulSoup

# Expect: exported_html_docs = [
#   {"url": ..., "file_id": ..., "name": ..., "html": "<!doctype html>..."},
#   ...
# ]

# Check if the html_docs is available
try:
    exported_html_docs  # list built in Step 5
except NameError:
    raise RuntimeError("`exported_html_docs` not found. Run the export step first.")

# Import function for extracting images from html_text
## Reload functions module for updates
import importlib, functions
importlib.reload(functions) 
from functions import extract_images_from_html_text

images_each_doc = []   # [{name, url, file_id, images:[{...}, ...]}, ...]
total_imgs = 0

for item in exported_html_docs:
    html_text = item.get("html", "") or ""
    imgs = extract_images_from_html_text(html_text)
    images_each_doc.append({
        "name": item.get("name"),
        "url": item.get("url"),
        "file_id": item.get("file_id"),
        "images": imgs,
    })
    total_imgs += len(imgs)

print(f"Extracted {total_imgs} <img> tag(s) across {len(images_each_doc)} HTML document(s).")
images_each_doc


Extracted 6 <img> tag(s) across 1 HTML document(s).


[{'name': 'Bài viết | 36 - kệ trung tải 4 tầng',
  'url': 'https://docs.google.com/document/d/19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E/edit?usp=sharing',
  'file_id': '19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E',
  'images': [{'alt': 'Kệ trung tải 4 tầng',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXf5nuKK8uXK9rFPUTDoTmbTpqXWHNnoOSATBAuOqExmiYAV-agJeBADf-Fkn5uEspm2T_nyxYBkLeSUDHbCOkBLVLPGxVJutEtclh6I_bSu_8dbYV1ykJBt4TwKXlHYfX0miXJhaA?key=5axpOQhltCUHo8WHcYAGCg',
    'style': 'width: 601.70px; height: 450.67px; margin-left: 0.00px; margin-top: 0.00px; transform: rotate(0.00rad) translateZ(0px); -webkit-transform: rotate(0.00rad) translateZ(0px);',
    'title': ''},
   {'alt': 'Bản vẽ chi tiết kệ',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXdPaYbaOtbV4r3iODPVJngIE4VD02QempFwvyavH3RqBqJwmJu8Wg1vxvfe6xuLmVng6dUR8SUnlWVx-Zt5M7wixGY0ji5W3AvYzNre-CQxx8RifFbRfReu0h6Gf8RImP9jaUcJbA?key=5axpOQhltCUHo8WHcYAGCg',
    'style': 'width: 601.70px; height: 450.6

In [51]:
## Step 8: Generate images slug and 5-word slug
try:
    images_each_doc  # from Step 9
except NameError:
    raise RuntimeError("`images_each_doc` not found. Run the image extraction step first.")

import importlib, functions
importlib.reload(functions) 
from functions import strip_diacritics
from functions import slugify
from functions import first_n_words

images_with_slugs = []  # [{doc_name, url, file_id, images:[{src, alt, slug_alt_full, slug_alt_first5, ...}]}]

for doc in images_each_doc:
    doc_entry = {
        "doc_name": doc.get("name"),
        "url": doc.get("url"),
        "file_id": doc.get("file_id"),
        "images": []
    }
    for attrs in doc.get("images", []):
        alt = attrs.get("alt", "") or ""
        slug_alt_full = slugify(alt)
        slug_alt_first5 = slugify(first_n_words(alt, 5))
        # copy existing attrs and add slugs
        enriched = dict(attrs)
        enriched["slug_alt_full"] = slug_alt_full
        enriched["slug_alt_first5"] = slug_alt_first5
        doc_entry["images"].append(enriched)
    images_with_slugs.append(doc_entry)

# Quick peek
total_imgs = sum(len(d["images"]) for d in images_with_slugs)
print(f"Added slugs for {total_imgs} image(s) across {len(images_with_slugs)} document(s).")
# Example:
# for d in images_with_slugs[:1]:
#     print(d["doc_name"], "→", len(d["images"]), "images")
#     for x in d["images"][:3]:
#         print(x.get("alt"), "=>", x["slug_alt_full"], "|", x["slug_alt_first5"])
images_with_slugs

Added slugs for 6 image(s) across 1 document(s).


[{'doc_name': 'Bài viết | 36 - kệ trung tải 4 tầng',
  'url': 'https://docs.google.com/document/d/19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E/edit?usp=sharing',
  'file_id': '19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E',
  'images': [{'alt': 'Kệ trung tải 4 tầng',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXf5nuKK8uXK9rFPUTDoTmbTpqXWHNnoOSATBAuOqExmiYAV-agJeBADf-Fkn5uEspm2T_nyxYBkLeSUDHbCOkBLVLPGxVJutEtclh6I_bSu_8dbYV1ykJBt4TwKXlHYfX0miXJhaA?key=5axpOQhltCUHo8WHcYAGCg',
    'style': 'width: 601.70px; height: 450.67px; margin-left: 0.00px; margin-top: 0.00px; transform: rotate(0.00rad) translateZ(0px); -webkit-transform: rotate(0.00rad) translateZ(0px);',
    'title': '',
    'slug_alt_full': 'ke-trung-tai-4-tang',
    'slug_alt_first5': 'ke-trung-tai-4-tang'},
   {'alt': 'Bản vẽ chi tiết kệ',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXdPaYbaOtbV4r3iODPVJngIE4VD02QempFwvyavH3RqBqJwmJu8Wg1vxvfe6xuLmVng6dUR8SUnlWVx-Zt5M7wixGY0ji5W3AvYzNre-CQxx8RifFbRfR

In [52]:
## Step 09: Download images using images_with_slugs

#Check if images_with_slugs is empty
try:
    images_with_slugs  # [{doc_name, url, file_id, images:[{src, alt, slug_alt_full, slug_alt_first5, ...}]}]
except NameError:
    raise RuntimeError("`images_with_slugs` not found. Run the slug-building step first.")

RAW_DIR = "images_raw"
os.makedirs(RAW_DIR, exist_ok=True)

downloaded_image_paths: dict[str, str] = {}
download_errors: list[dict] = []


importlib.reload(functions)                 #Reload functions file for updates
from functions import safe_name             #Get names for saving iamges
from functions import ensure_unique_path    #Ensure images have unique path
from functions import guess_ext_from_url    #Take the file type from url
from functions import download_to_file      #Download images

total_attempts = 0
print("\n--- Downloading images from images_with_slugs ---")

for doc in images_with_slugs:
    doc_name = safe_name(doc.get("doc_name") or doc.get("file_id") or "document")
    out_dir = os.path.join(RAW_DIR, doc_name)
    os.makedirs(out_dir, exist_ok=True)

    imgs = doc.get("images", []) or []
    if not imgs:
        continue

    print(f"\n{doc_name}: {len(imgs)} image(s)")

    for i, attrs in enumerate(imgs, start=1):
        src = attrs.get("src")
        if not src:
            continue

        # prefer short slug; fallback to full; fallback to generic
        base = (attrs.get("slug_alt_first5") or attrs.get("slug_alt_full") or f"image-{i}").strip("-") or f"image-{i}"
        ext  = guess_ext_from_url(src)
        filename = f"{base}{ext}"
        dst = ensure_unique_path(out_dir, filename)

        total_attempts += 1
        try:
            print(f"  ↓ {src}\n    → {dst}")
            download_to_file(src, dst, timeout=60)
            downloaded_image_paths[src] = dst
        except Exception as e:
            download_errors.append({"src": src, "filename": filename, "doc_name": doc_name, "error": str(e)})
            print(f"    ✗ Failed: {e}")

print("\n--- Download summary ---")
print(f"Total attempted: {total_attempts}")
print(f"Successfully downloaded: {len(downloaded_image_paths)}")
print(f"Failed: {len(download_errors)}")



--- Downloading images from images_with_slugs ---

Bài viết _ 36 - kệ trung tải 4 tầng: 6 image(s)
  ↓ https://lh7-rt.googleusercontent.com/docsz/AD_4nXf5nuKK8uXK9rFPUTDoTmbTpqXWHNnoOSATBAuOqExmiYAV-agJeBADf-Fkn5uEspm2T_nyxYBkLeSUDHbCOkBLVLPGxVJutEtclh6I_bSu_8dbYV1ykJBt4TwKXlHYfX0miXJhaA?key=5axpOQhltCUHo8WHcYAGCg
    → images_raw/Bài viết _ 36 - kệ trung tải 4 tầng/ke-trung-tai-4-tang_11.jpg
  ↓ https://lh7-rt.googleusercontent.com/docsz/AD_4nXdPaYbaOtbV4r3iODPVJngIE4VD02QempFwvyavH3RqBqJwmJu8Wg1vxvfe6xuLmVng6dUR8SUnlWVx-Zt5M7wixGY0ji5W3AvYzNre-CQxx8RifFbRfReu0h6Gf8RImP9jaUcJbA?key=5axpOQhltCUHo8WHcYAGCg
    → images_raw/Bài viết _ 36 - kệ trung tải 4 tầng/ban-ve-chi-tiet-ke_11.jpg
  ↓ https://lh7-rt.googleusercontent.com/docsz/AD_4nXehGr9gXi5FcxADzfzqAUOpJowZHl-VOu6j-XTRC-ly0-RZtoAlPj0_1zk_EZw4-7p_gKu7dWjobQgtHFB0_LE9HpV65rG6zaDLxq-92zqAEzInJlo_3MrC3AI6duHDX7K_VMBX?key=5axpOQhltCUHo8WHcYAGCg
    → images_raw/Bài viết _ 36 - kệ trung tải 4 tầng/thiet-ke-ke-4-tang_11.jpg
  ↓ https://l

In [53]:
## Step 10: Resize the images
# Define target width and height
target_image_width = 800  # Example width in pixels
target_image_height = 600 # Example height in pixels

# Check if download images are available
try:
    downloaded_image_paths
except NameError:
    raise RuntimeError("`downloaded_image_paths` not found. Run the download step first.")

# Define the path for saving resized images
RESIZED_DIR = "images_resized"
os.makedirs(RESIZED_DIR, exist_ok=True)

# Create dict for storing src_url
resized_image_paths = {}   # {src_url: resized_local_path}
resize_errors = []

# Import necessary function
importlib.reload(functions)          
from functions import resize_fit

# Resize images one by one in downloaded image
for src_url, local_path in downloaded_image_paths.items():
    try:
        out_path = os.path.join(RESIZED_DIR, os.path.basename(local_path))
        resize_fit(local_path, out_path, target_image_width, target_image_height)
        resized_image_paths[src_url] = out_path
        print(f"Resized → {out_path}")
    except (UnidentifiedImageError, OSError) as e:
        resize_errors.append({"source_url": src_url, "path": local_path, "error": str(e)})
        print(f"Failed to resize {local_path}: {e}")

print(f"\nResize summary: resized={len(resized_image_paths)} failed={len(resize_errors)}")


Resized → images_resized/ke-trung-tai-4-tang_11.jpg
Resized → images_resized/ban-ve-chi-tiet-ke_11.jpg
Resized → images_resized/thiet-ke-ke-4-tang_11.jpg
Resized → images_resized/uu-iem-ke-4-tang_11.jpg
Resized → images_resized/ung-dung-ke-4-tang_11.jpg
Resized → images_resized/ke-4-tang-rackcosmo_11.jpg

Resize summary: resized=6 failed=0


In [ ]:
## Step 11: Setup Wordpress API

# Set up Wordpress API
WP_BASE_URL = "https://ngoncareer.com"          # No trailing slash
WP_USERNAME  = "admin_career"
wp_pass_clean = (WP_APP_PASS or "").replace(" ", "")  # WP_APP_PASS defined in environment

# Check if API is missing some variables
if not WP_BASE_URL or not WP_USERNAME or not wp_pass_clean:
    raise ValueError("Missing WP_BASE_URL / WP_USERNAME / WP_APP_PASS")

# Build the REST base and commonly used endpoints
WP_API_BASE = f"{WP_BASE_URL}/wp-json/wp/v2"
WP_MEDIA_EP = f"{WP_API_BASE}/media"

# Build Basic Auth header
token = base64.b64encode(f"{WP_USERNAME}:{wp_pass_clean}".encode("utf-8")).decode("utf-8")
auth_header = {"Authorization": f"Basic {token}"}

# Final_check
print("WordPress REST API auth header ready.")
print("Base:", WP_API_BASE)

In [ ]:
#Step 12: Upload images to wordpress

#Check if the API endpoint, API authentication, resized images are all available
try:
    WP_MEDIA_EP, auth_header, resized_image_paths
except NameError:
    raise RuntimeError("Missing WP_MEDIA_EP/auth_header or resized_image_paths.")

#Import necessary functions
importlib.reload(functions)
from functions import guess_mime        #Get file extension 
from functions import upload_resized    #Uploading resized images

# Results:
# - wp_uploaded_map: {original_src_url: {"id": <media_id>, "wp_url": <public_url>, "name": <title>}}
# - wp_upload_errors: [{source_url, local_path, error}]
wp_uploaded_map = {}
wp_upload_errors = []

print("\n— Uploading RESIZED images to WordPress —")
for orig_src, local_path in resized_image_paths.items():
    try:
        media = upload_resized(local_path)
        wp_uploaded_map[orig_src] = {
            "id": media.get("id"),
            "wp_url": media.get("source_url"),
            "name": (media.get("title") or {}).get("rendered") or media.get("slug") or os.path.basename(local_path),
            "mime_type": media.get("mime_type"),
            "local_path": local_path,
        }
        print(f"✓ {os.path.basename(local_path)} → {wp_uploaded_map[orig_src]['wp_url']}")
    except Exception as e:
        wp_upload_errors.append({"source_url": orig_src, "local_path": local_path, "error": str(e)})
        print(f"✗ Failed: {local_path} | {e}")

print(f"\nUpload summary: uploaded={len(wp_uploaded_map)} | failed={len(wp_upload_errors)}")
# Now you can use `wp_uploaded_map[original_src]["wp_url"]` to update images_with_slugs later.


In [ ]:
## Step 13: Update images_with_slugs variable
## Step: Clean + update images_with_slugs
# - Ensure no pre-existing `width` / `height` on each image, then add fresh values
# - For `new_src`: if it exists, remove it first; then set it to the new WordPress URL

import os

# Pillow for reading actual resized dimensions
try:
    from PIL import Image
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pillow"])
    from PIL import Image

# Requires:
# - images_with_slugs: [{doc_name, url, file_id, images:[{src, ...}]}]
# - resized_image_paths: {original_src_url: "/path/to/resized.ext"}
# - wp_uploaded_map: {original_src_url: {"id":..., "wp_url":..., "local_path":...}}

try:
    images_with_slugs, resized_image_paths, wp_uploaded_map
except NameError:
    raise RuntimeError("Missing images_with_slugs, resized_image_paths, or wp_uploaded_map.")

updated = 0
skipped = 0
errors = []

for doc in images_with_slugs:
    for img in doc.get("images", []):
        orig_src = img.get("src")
        if not orig_src:
            skipped += 1
            continue

        up = wp_uploaded_map.get(orig_src)
        local_resized = resized_image_paths.get(orig_src)

        if not up or not local_resized or not os.path.isfile(local_resized):
            # No upload info or no resized file → cannot update
            skipped += 1
            continue

        # --- 1) remove any existing width/height first
        img.pop("width", None)
        img.pop("height", None)

        # --- 2) remove existing new_src (if present), then add new one
        img.pop("new_src", None)
        img["new_src"] = up.get("wp_url")

        # --- 3) compute actual resized dimensions and add width/height
        try:
            with Image.open(local_resized) as im:
                w, h = im.size
            img["width"] = int(w)
            img["height"] = int(h)
            # optional helpful fields:
            img["uploaded_media_id"] = up.get("id")
            img["resized_local_path"] = local_resized
            updated += 1
        except Exception as e:
            errors.append({"src": orig_src, "path": local_resized, "error": str(e)})

print(f"Cleaned & updated {updated} image(s). Skipped: {skipped}. Errors: {len(errors)}")
# Optional quick peek:
# [ (img.get('src'), img.get('new_src'), img.get('width'), img.get('height'))
#   for d in images_with_slugs for img in d['images'][:3] ]
images_with_slugs

In [ ]:
## Step 14: Extract the main heading from each exported HTML
## Step: Extract ONLY the first <h1> from each exported HTML

try:
    exported_html_docs  # [{url, file_id, name, html}]
except NameError:
    raise RuntimeError("`exported_html_docs` not found. Run the export step first.")

def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())

h1_summary = []  # [{file_id, name, h1}]

for doc in exported_html_docs:
    html = doc.get("html", "") or ""
    soup = BeautifulSoup(html, "html.parser")

    # only the FIRST <h1>
    h1_tag = soup.find("h1")
    h1_text = clean_text(h1_tag.get_text()) if h1_tag and h1_tag.get_text(strip=True) else None

    # save back to the doc
    doc["h1"] = h1_text

    h1_summary.append({
        "file_id": doc.get("file_id"),
        "name": doc.get("name"),
        "h1": h1_text,
    })

    if h1_text is None:
        print(f"(no <h1>) {doc.get('name') or doc.get('file_id')}")

print(f"Extracted first <h1> for {len(h1_summary)} document(s).")
# Optional peek:
# for row in h1_summary[:5]:
print(h1_summary[0]["h1"])


In [ ]:
##Step 15: Create empty post in wordpress
# Requires:
# - WP_API_BASE and auth_header from your Step 11
# - h1_summary (list of dicts with key "h1")
# - main_keyword_slug (string)

try:
    WP_API_BASE, auth_header, h1_summary, main_keyword_slug_to_run
except NameError:
    raise RuntimeError("Missing WP_API_BASE/auth_header or h1_summary/main_keyword_slug_to_run.")

posts_endpoint = f"{WP_API_BASE}/posts"
title = (h1_summary[0].get("h1") or "").strip()

post_data = {
    "title": title,
    "slug": main_keyword_slug_to_run[0],
    "status": "draft",
    "content": "",
}

print(f"Creating draft post at: {posts_endpoint}")
resp = requests.post(posts_endpoint, headers=auth_header, json=post_data)

if resp.status_code in (200, 201):
    data = resp.json()
    wordpress_post_id = data.get("id")
    wordpress_post_link = data.get("link")
    print(f"✓ Draft created. ID: {wordpress_post_id} | Link: {wordpress_post_link}")
else:
    raise RuntimeError(f"Post create failed {resp.status_code}: {resp.text[:500]}")


In [ ]:
##Step 16: Update images metadata
# Requires:
#   - WP_API_BASE or WP_MEDIA_EP, and auth_header (from Step 11)
#   - wordpress_post_id (created post ID)
#   - wp_uploaded_map: {original_src_url: {"id": <media_id>, "wp_url": <media_url>, ...}}
# Optional:
#   - images_with_slugs to supply alt text per original src

# ---- inputs check ----
try:
    wordpress_post_id
    auth_header
    WP_API_BASE
except NameError:
    raise RuntimeError("Missing wordpress_post_id, auth_header, or WP_API_BASE (from Step 11).")

WP_MEDIA_EP = f"{WP_API_BASE}/media"

try:
    wp_uploaded_map
except NameError:
    raise RuntimeError("Missing `wp_uploaded_map` (built after uploads).")

# Build src_url -> alt text map from images_with_slugs (optional)
src_to_alt = {}
if "images_with_slugs" in globals() and isinstance(images_with_slugs, list):
    for doc in images_with_slugs:
        for img in doc.get("images", []):
            src = img.get("src")
            if not src:
                continue
            alt = (img.get("alt") or "").strip()
            # first win
            src_to_alt.setdefault(src, alt)

updated, failed = 0, []
print("\n— Updating media metadata and associating with the post —")

for original_src, meta in wp_uploaded_map.items():
    media_id = meta.get("id")
    if not media_id:
        failed.append({"src": original_src, "error": "Missing media_id"})
        continue

    alt = src_to_alt.get(original_src, "").strip()

    payload = {
        "alt_text": alt,                 # alt on image
        "caption": alt,                  # caption text
        "description": alt,              # description text
        "post": int(wordpress_post_id),  # attach to the created post
    }

    try:
        r = requests.post(
            f"{WP_MEDIA_EP}/{media_id}",
            headers={**auth_header, "Content-Type": "application/json"},
            json=payload,
            timeout=60,
        )
        if r.status_code >= 400:
            raise RuntimeError(f"{r.status_code}: {r.text[:400]}")
        updated += 1
        print(f"✓ Media {media_id} updated")
    except Exception as e:
        failed.append({"media_id": media_id, "src": original_src, "error": str(e)})
        print(f"✗ Media {media_id} failed: {e}")

print(f"\nSummary: updated={updated} | failed={len(failed)}")
# Optional peek:
# failed[:3]

# Append the WordPress post_id onto every image entry in images_with_slugs
## Append/refresh WordPress post_id on images_with_slugs (no duplicates)

try:
    wordpress_post_id
    images_with_slugs
except NameError:
    raise RuntimeError("Missing `wordpress_post_id` or `images_with_slugs`.")

pid = int(wordpress_post_id)
set_count = 0
already_count = 0
corrected_count = 0

for doc in images_with_slugs:
    for img in doc.get("images", []):
        if "post_id" in img:
            # If it's different (or wrong type), refresh it; otherwise leave as-is
            if img["post_id"] != pid:
                img["post_id"] = pid
                corrected_count += 1
            else:
                already_count += 1
        else:
            img["post_id"] = pid
            set_count += 1

print(
    f"post_id updates → set:{set_count}, corrected:{corrected_count}, already-correct:{already_count}"
)
images_with_slugs

In [ ]:
# Step 17 (VS Code): Replace <img> tags with WordPress [caption] shortcodes (src + alt + width + height)
# Strategy:
#   1) Try exact per-doc match by original `src` from images_with_slugs.
#   2) If no exact match (Docs regenerates URLs), fall back to order: 1st <img> ↔ 1st image data, etc.

from bs4 import BeautifulSoup, NavigableString
from html import unescape as html_unescape  # avoid shadowing the html module

# Requires:
# - exported_html_docs: [{file_id, name, html, ...}]
# - images_with_slugs : [{file_id or doc_name/name/url, images:[{src, new_src/new_url, alt, width/new_width, height/new_height, uploaded_media_id}, ...]}]

try:
    exported_html_docs, images_with_slugs
except NameError:
    raise RuntimeError("Missing `exported_html_docs` or `images_with_slugs`. Run previous steps first.")

DEFAULT_ALIGN = "aligncenter"
DEFAULT_SIZE_CLASS = "size-full"

# Group images_with_slugs by file_id (preferred) and by name (fallback)
by_file_id, by_name = {}, {}
for d in images_with_slugs:
    key_id = d.get("file_id")
    key_name = d.get("doc_name") or d.get("name") or d.get("url")
    images = d.get("images", []) or []
    if key_id:
        by_file_id.setdefault(key_id, []).extend(images)
    if key_name:
        by_name.setdefault(key_name, []).extend(images)

#Import functions
importlib.reload(functions)
from functions import build_caption_shortcode

processed_html_docs = []   # [{file_id, name, html_processed, images_replaced}]
total_replaced = 0

for doc in exported_html_docs:
    file_id = doc.get("file_id")
    name    = doc.get("name")
    html_in = doc.get("html") or ""

    soup = BeautifulSoup(html_in, "html.parser")
    img_tags = soup.find_all("img")

    # per-doc image data
    imgs_list = by_file_id.get(file_id)
    if imgs_list is None:
        imgs_list = by_name.get(name, [])
    per_doc_src_map = { img.get("src"): img for img in imgs_list if img.get("src") }

    n_html = len(img_tags)
    n_data = len(imgs_list)
    n_replace = min(n_html, n_data)  # fallback upper-bound when no exact match
    replaced = 0
    idx_fallback = 0

    for i, tag in enumerate(img_tags):
        orig_src = tag.get("src")
        matched_info = None

        # 1) exact per-doc src match
        if orig_src in per_doc_src_map:
            matched_info = per_doc_src_map[orig_src]

        # 2) fallback by position if still no match
        if matched_info is None and idx_fallback < n_data:
            matched_info = imgs_list[idx_fallback]
            idx_fallback += 1

        if not matched_info:
            continue
        if not (matched_info.get("new_src") or matched_info.get("new_url")):
            continue

        shortcode = build_caption_shortcode(matched_info)
        tag.replace_with(NavigableString(shortcode))
        replaced += 1

    processed_html_docs.append({
        "file_id": file_id,
        "name": name,
        "html_processed": html_unescape(str(soup)),  # ensure shortcodes not HTML-escaped
        "images_replaced": replaced,
    })
    total_replaced += replaced

print(f"Processed {len(processed_html_docs)} document(s). Total images replaced: {total_replaced}")

# Optional: handy dict for further steps (e.g., updating WP post content)
processed_html_contents = { (d["file_id"] or d["name"]): d["html_processed"] for d in processed_html_docs }


In [ ]:
# Step 18: Robust HTML transforms (VS Code friendly)
from bs4 import BeautifulSoup, NavigableString
from urllib.parse import urlparse, parse_qs, unquote
import re

#Import functions
importlib.reload(functions)
from functions import _parse_style
from functions import _style_to_str
from functions import transform_html_dom

In [ ]:
# Step 19: Apply changes

#Check if the html content is available
try:
    processed_html_docs
except NameError:
    raise RuntimeError("Run the image-replacement step first to build `processed_html_docs`.")

changed = 0
for d in processed_html_docs:
    src = d.get("html_processed") or d.get("html") or ""
    out = transform_html_dom(src)
    if out != src:
        changed += 1
    d["html_processed"] = out

print(f"Transformed {changed}/{len(processed_html_docs)} document(s).")

In [ ]:
## Step 20: Update content of post in wordpress

# Requires:
# - WP_API_BASE, auth_header (from Step 11)
# - wordpress_post_id (from Step 15)
# - processed_html_docs (from the replace step)

try:
    WP_API_BASE, auth_header, wordpress_post_id, processed_html_docs
except NameError:
    raise RuntimeError("Missing WP_API_BASE/auth_header, wordpress_post_id, or processed_html_docs.")

if not processed_html_docs:
    raise RuntimeError("processed_html_docs is empty. Run the HTML processing step first.")

# Choose which processed doc to insert (here: the first one)
html_str = processed_html_docs[0]["html_processed"]

posts_endpoint = f"{WP_API_BASE}/posts/{int(wordpress_post_id)}"
payload = {
    "content": html_str,   # raw HTML (can include shortcodes)
    # "status": "draft",   # optional: keep as draft
    # "status": "publish"  # optional: publish immediately
}

print(f"Updating post #{wordpress_post_id} at: {posts_endpoint}")
resp = requests.post(
    posts_endpoint,
    headers={**auth_header, "Content-Type": "application/json"},
    json=payload,
    timeout=90,
)

if resp.status_code == 200:
    data = resp.json()
    print(f"✓ Post updated. View: {data.get('link')}")
else:
    raise RuntimeError(f"Update failed {resp.status_code}: {resp.text[:500]}")
